# CE 1110 - Tarea 1: Generación y caracterización de señales

Genera tres casos reproducibles (una semilla distinta por caso), verifica la SNR obtenida, caracteriza las señales en tiempo, magnitud, fase y energía por bandas, y exporta WAV, metadatos y métricas.

Basado en el notebook `CE1110_generador_tonos_defensa_Kaggle.ipynb` del profesor. Las funciones están en `src/generacion_senales.py`.

In [1]:
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = Path.cwd()
sys.path.insert(0, str(RAIZ / 'src'))
import generacion_senales as gs

plt.style.use('seaborn-v0_8-whitegrid')
print('Entorno listo. Raíz del proyecto:', RAIZ)

Entorno listo. Raíz del proyecto: /home/fabs/Github/jgarcia_asm_2026_s2


## 1. Configuración de los tres casos
Parámetros comunes: fs = 16 kHz, 6 s (N = 96 000 muestras), multitono 440/880/1320 Hz. SNR objetivo = 5 dB en los tres casos: la interferencia queda ~3.6 dB por debajo del tono más fuerte, lo bastante exigente sin que la señal útil quede enterrada.

| Caso | Ruido | Relación con la banda útil |
|---|---|---|
| 1 | Interferencia sinusoidal a 1750 Hz | Separada de 440, 880 y 1320 Hz |
| 2 | Ruido blanco | Ocupa todo el espectro, pero la energía por bin es baja |
| 3 | Ruido de banda limitada 700-1500 Hz | Se superpone a los tonos de 880 y 1320 Hz |

In [2]:
BASE = dict(
    fs_Hz=16_000, duracion_s=6.0, pico_maximo=0.80, fade_ms=20,
    tipo_senal='multitono',
    frecuencias_utiles_Hz=[440, 880, 1320],
    amplitudes_utiles=[1.0, 0.55, 0.30],
    fases_grados=[0, 35, -50],
    snr_objetivo_dB=5.0,
    numero_impulsos=10,
    frecuencia_interferencia_Hz=1750,
    banda_ruido_Hz=[2500, 3800],
)

CASOS = {
    'caso_1': dict(BASE, nombre='caso_1_interferencia_sinusoidal', semilla=20261001,
                   tipo_ruido='tonal', frecuencia_interferencia_Hz=1750),
    'caso_2': dict(BASE, nombre='caso_2_ruido_blanco', semilla=20261002,
                   tipo_ruido='blanco'),
    'caso_3': dict(BASE, nombre='caso_3_banda_superpuesta', semilla=20261003,
                   tipo_ruido='banda', banda_ruido_Hz=[700, 1500]),
}
pd.DataFrame(CASOS).T[['semilla','tipo_ruido','snr_objetivo_dB','frecuencia_interferencia_Hz','banda_ruido_Hz']]

,semilla,tipo_ruido,snr_objetivo_dB,frecuencia_interferencia_Hz,banda_ruido_Hz
caso_1,20261001,tonal,5.0,1750,"[2500, 3800]"
caso_2,20261002,blanco,5.0,1750,"[2500, 3800]"
caso_3,20261003,banda,5.0,1750,"[700, 1500]"


## 2. Generación y verificación de la SNR

In [3]:
RES = {k: gs.generar_caso(cfg) for k, cfg in CASOS.items()}

filas = []
for k, cfg in CASOS.items():
    r = RES[k]
    err = r['snr_obtenida_dB'] - cfg['snr_objetivo_dB']
    assert abs(err) < 1e-3, f'{k}: SNR fuera de tolerancia ({err:.2e} dB)'
    filas.append({
        'caso': k, 'fs_Hz': cfg['fs_Hz'], 'N': len(r['t']), 'duracion_s': cfg['duracion_s'],
        'frecuencias_utiles_Hz': cfg['frecuencias_utiles_Hz'], 'tipo_ruido': cfg['tipo_ruido'],
        'semilla': cfg['semilla'], 'SNR_objetivo_dB': cfg['snr_objetivo_dB'],
        'SNR_obtenida_dB': round(r['snr_obtenida_dB'], 6), 'error_dB': f'{err:.2e}',
    })
TABLA_PARAMS = pd.DataFrame(filas).set_index('caso')
TABLA_PARAMS

,fs_Hz,N,duracion_s,frecuencias_utiles_Hz,tipo_ruido,semilla,SNR_objetivo_dB,SNR_obtenida_dB,error_dB
caso,,,,,,,,,
caso_1,16000,96000,6.0,"[440, 880, 1320]",tonal,20261001,5.0,5.0,1.33e-14
caso_2,16000,96000,6.0,"[440, 880, 1320]",blanco,20261002,5.0,5.0,7.99e-15
caso_3,16000,96000,6.0,"[440, 880, 1320]",banda,20261003,5.0,5.0,2.84e-14


La SNR obtenida coincide con la objetivo (el `assert` exige error < 0.001 dB); la diferencia residual es de redondeo en coma flotante. La normalización a pico 0.8 usa una ganancia común para señal, ruido y mezcla, por lo que no altera la SNR.

## 3. Señal temporal

In [4]:
FIGS = RAIZ / 'resultados' / 'figuras'
FIGS.mkdir(parents=True, exist_ok=True)

fig, axs = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for ax, (k, cfg) in zip(axs, CASOS.items()):
    r = RES[k]; fs = cfg['fs_Hz']; v = int(0.02*fs)
    ax.plot(r['t'][:v]*1e3, r['clean'][:v], label='Limpia', lw=1.6)
    ax.plot(r['t'][:v]*1e3, r['noise'][:v], label='Ruido', lw=0.8, alpha=.8)
    ax.plot(r['t'][:v]*1e3, r['noisy'][:v], label='Contaminada', lw=0.8, alpha=.8)
    ax.set(ylabel='Amplitud (normalizada)', title=f"{k}: {cfg['tipo_ruido']} (SNR = {r['snr_obtenida_dB']:.2f} dB)")
    ax.legend(loc='upper right', ncol=3)
axs[-1].set_xlabel('Tiempo (ms)')
plt.tight_layout(); plt.savefig(FIGS/'2_2_senal_temporal.png', dpi=150); plt.show()

/tmp/ipykernel_44933/1558242443.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIGS/'2_2_senal_temporal.png', dpi=150); plt.show()


## 4. Magnitud y fase del espectro

In [5]:
fig, axs = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
for ax, (k, cfg) in zip(axs, CASOS.items()):
    r = RES[k]; fs = cfg['fs_Hz']
    f, mc = gs.espectro(r['clean'], fs); _, mn = gs.espectro(r['noisy'], fs)
    ax.plot(f, mn, lw=.7, label='Contaminada', alpha=.8)
    ax.plot(f, mc, lw=.9, label='Limpia')
    for fu in cfg['frecuencias_utiles_Hz']: ax.axvline(fu, color='g', ls=':', lw=.8)
    ax.set(ylabel='Magnitud (dB)', ylim=(-100, 5), title=k); ax.legend(loc='upper right')
axs[-1].set_xlabel('Frecuencia (Hz)'); axs[-1].set_xlim(0, 4000)
plt.tight_layout(); plt.savefig(FIGS/'2_2_magnitud.png', dpi=150); plt.show()

/tmp/ipykernel_44933/3497097484.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIGS/'2_2_magnitud.png', dpi=150); plt.show()


In [6]:
fig, axs = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for ax, (k, cfg) in zip(axs, CASOS.items()):
    r = RES[k]; fs = cfg['fs_Hz']
    f, pc = gs.fase_espectro(r['clean'], fs); _, pn = gs.fase_espectro(r['noisy'], fs)
    ax.plot(f, pn, '.', ms=2, label='Contaminada', alpha=.6)
    ax.plot(f, pc, 'o', ms=4, mfc='none', label='Limpia')
    ax.set(ylabel='Fase (grados)', ylim=(-190, 190), title=f'{k}: fase (solo bins con |X| > 0.1 % del máximo)')
    ax.legend(loc='upper right')
axs[-1].set_xlabel('Frecuencia (Hz)'); axs[-1].set_xlim(0, 4000)
plt.tight_layout(); plt.savefig(FIGS/'2_2_fase.png', dpi=150); plt.show()

/tmp/ipykernel_44933/197827461.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIGS/'2_2_fase.png', dpi=150); plt.show()


**Lectura de la fase:** en el bin de cada tono la fase de la señal limpia es la fase configurada menos 90° (el seno se mide respecto al coseno de la FFT): -90°, -55° y -140° para 440, 880 y 1320 Hz. Los puntos alrededor de cada tono son bins vecinos con fuga espectral, causada por el fade de 20 ms, y están desfasados 180° respecto al bin central.

- **Caso 1:** los tres tonos conservan su fase. La interferencia aparece como un único punto nuevo en 1750 Hz.
- **Caso 2:** el ruido blanco rellena todo el espectro con fases aleatorias uniformes en ±180°. Los tonos siguen dominando en su bin, por eso conservan la fase.
- **Caso 3:** la fase aleatoria queda confinada a 700-1500 Hz. En los bins exactos de 880 y 1320 Hz la fase del tono sigue visible, porque la energía del tono se concentra en un bin; pero los bins vecinos, con fase distorsionada, ya no son separables del ruido.

## 5. Distribución de energía por bandas

In [7]:
BORDES = np.arange(0, 8001, 500)  # 16 bandas de 500 Hz hasta Nyquist
centros = (BORDES[:-1] + BORDES[1:]) / 2

fig, axs = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
for ax, (k, cfg) in zip(axs, CASOS.items()):
    r = RES[k]; fs = cfg['fs_Hz']
    ec = gs.energia_por_bandas(r['clean'], fs, BORDES) * 100
    en = gs.energia_por_bandas(r['noise'], fs, BORDES) * 100
    ax.bar(centros-90, ec, width=170, label='Limpia')
    ax.bar(centros+90, en, width=170, label='Ruido')
    ax.set(ylabel='% de energía de cada señal', yscale='symlog', title=k); ax.legend(loc='upper right')
axs[-1].set_xlabel('Frecuencia (Hz), bandas de 500 Hz')
plt.tight_layout(); plt.savefig(FIGS/'2_2_energia_bandas.png', dpi=150); plt.show()

/tmp/ipykernel_44933/3805951083.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIGS/'2_2_energia_bandas.png', dpi=150); plt.show()


In [8]:
from scipy.signal import welch
fig, axs = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for ax, (k, cfg) in zip(axs, CASOS.items()):
    r = RES[k]; fs = cfg['fs_Hz']
    for nombre, x in [('Limpia', r['clean']), ('Ruido', r['noise']), ('Contaminada', r['noisy'])]:
        f, p = welch(x, fs=fs, nperseg=4096)
        ax.semilogy(f, p, label=nombre, lw=.9)
    ax.set(ylabel='DEP (V²/Hz)', title=f'{k}: densidad espectral de potencia (Welch)'); ax.legend(loc='upper right')
axs[-1].set_xlabel('Frecuencia (Hz)')
plt.tight_layout(); plt.savefig(FIGS/'2_2_dep_welch.png', dpi=150); plt.show()

/tmp/ipykernel_44933/4017764372.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIGS/'2_2_dep_welch.png', dpi=150); plt.show()


## 6. Características para detectar automáticamente el ruido

In [9]:
filas = []
for k, cfg in CASOS.items():
    r = RES[k]
    for fuente, x in [('ruido_aislado', r['noise']), ('contaminada', r['noisy'])]:
        d = gs.caracteristicas_ruido(x, cfg['fs_Hz'])
        filas.append(dict(caso=k, tipo_ruido=cfg['tipo_ruido'], fuente=fuente, **d))
FEATURES = pd.DataFrame(filas).set_index(['caso', 'fuente']).round(3)
FEATURES

tipo_ruido  curtosis_exceso  planicidad_espectral  \
caso   fuente                                                            
caso_1 ruido_aislado      tonal           -1.500                 0.000   
       contaminada        tonal           -0.768                 0.000   
caso_2 ruido_aislado     blanco            0.010                 0.988   
       contaminada       blanco           -0.694                 0.243   
caso_3 ruido_aislado      banda           -0.010                 0.000   
       contaminada        banda           -0.678                 0.000   

                      pico_sobre_mediana_dB  ancho_banda_95_Hz  
caso   fuente                                                   
caso_1 ruido_aislado                120.000              7.812  
       contaminada                  120.000           1316.406  
caso_2 ruido_aislado                  1.918           7589.844  
       contaminada                   34.271           6734.375  
caso_3 ruido_aislado                120.000            738.281  
       contaminada                  120.000            957.031

**Interpretación (sobre el ruido aislado, que es la referencia, y la señal contaminada, que es lo que ve el sistema):**

| Tipo | Curtosis en exceso | Planicidad espectral | Ancho de banda 95 % |
|---|---|---|---|
| Tonal (caso 1) | -1.5 (sinusoide) | ≈ 0 | ≈ 8 Hz |
| Blanco (caso 2) | ≈ 0 (gaussiano) | 0.99 | ≈ 7600 Hz |
| Banda (caso 3) | ≈ 0 (gaussiano) | ≈ 0 | ≈ 740 Hz |

- **Interferencia tonal:** la curtosis de -1.5 es propia de una sinusoide y su ancho de banda es de unos pocos Hz: es un pico aislado fuera de los tonos útiles.
- **Ruido blanco:** es el único con planicidad cercana a 1 y ancho de banda casi igual al disponible. La curtosis no lo distingue del ruido de banda, porque ambos son gaussianos. En la señal contaminada el piso de ruido queda elevado (pico sobre mediana de 34 dB, frente a valores saturados en 120 dB de los otros dos casos).
- **Ruido de banda limitada:** gaussiano pero con ancho de banda acotado (≈ 740 Hz), lo que lo separa del blanco. La planicidad total es baja porque la mayor parte del espectro está vacía.
- **En el sistema real** no hay referencia limpia, así que estos descriptores se estiman sobre bloques de la señal contaminada, donde los tonos útiles también aportan picos. Por eso conviene medir el piso de ruido entre los picos y no solo el máximo.

## 7. ¿Qué no se puede recuperar cuando señal y ruido comparten frecuencias? (caso 3)

En 700-1500 Hz hay tonos útiles (880 y 1320 Hz) y ruido. Un filtro o una máscara espectral solo puede actuar sobre la frecuencia, no distingue qué parte de la energía de un bin es señal y cuál es ruido. Por eso:

1. **Información irrecuperable:** en los bins donde coexisten señal y ruido, el ruido es una suma aditiva con fase aleatoria. Atenuar la banda para quitar ruido también atenúa y distorsiona amplitud y fase de los tonos de 880 y 1320 Hz; dejarla intacta conserva el ruido. No hay forma de separar ambos sin información adicional (modelo de la señal, referencia, etc.).
2. **Compromiso:** cuanto más energía del ruido se elimina, más energía útil se pierde (mayor MSE respecto a la señal limpia). Esto se mide en la sección 2.5.
3. **Qué sí se puede:** estimar y restar el nivel medio de ruido fuera de los picos de los tonos (por ejemplo, sustracción espectral), reduciendo el ruido pero sin eliminar la componente que cae en el mismo bin del tono.
4. **Caso contrario (casos 1 y 2):** con una interferencia separada se puede rechazar solo esa frecuencia sin tocar la señal útil; con ruido blanco, el ruido que cae fuera de los picos puede quitarse, pero el que cae bajo los tonos permanece.

## 8. Exportación (WAV, metadatos y métricas)

In [10]:
for k, cfg in CASOS.items():
    gs.exportar_caso(cfg, RES[k], RAIZ / 'pruebas' / k)

(RAIZ / 'metadatos').mkdir(exist_ok=True)
configuraciones = {k: dict(cfg, N=len(RES[k]['t']), snr_obtenida_dB=RES[k]['snr_obtenida_dB'],
                           ganancia_comun=float(RES[k]['ganancia'])) for k, cfg in CASOS.items()}
with open(RAIZ / 'metadatos' / 'configuraciones.json', 'w', encoding='utf-8') as f:
    json.dump(configuraciones, f, indent=2, ensure_ascii=False)

(RAIZ / 'resultados').mkdir(exist_ok=True)
TABLA_PARAMS.to_csv(RAIZ / 'resultados' / 'caracterizacion_senales.csv')
FEATURES.to_csv(RAIZ / 'resultados' / 'caracteristicas_ruido.csv')
print('Exportado:', *sorted(str(p.relative_to(RAIZ)) for p in (RAIZ/'pruebas').rglob('*') if p.is_file()), sep='\n  ')

Exportado:
  pruebas/caso_1/01_referencia_limpia.wav
  pruebas/caso_1/02_ruido.wav
  pruebas/caso_1/03_entrada_contaminada.wav
  pruebas/caso_1/metadatos_prueba.json
  pruebas/caso_2/01_referencia_limpia.wav
  pruebas/caso_2/02_ruido.wav
  pruebas/caso_2/03_entrada_contaminada.wav
  pruebas/caso_2/metadatos_prueba.json
  pruebas/caso_3/01_referencia_limpia.wav
  pruebas/caso_3/02_ruido.wav
  pruebas/caso_3/03_entrada_contaminada.wav
  pruebas/caso_3/metadatos_prueba.json
